In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver   

In [ ]:
load_dotenv() 

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [13]:
config1 = {'configurable':{'thread_id':'1'}}
workflow.invoke({'topic':"Cricket"},config=config1)

{'topic': 'Cricket',
 'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a "sticky" wicket situation and was feeling a little "bowled" over.',
 'explanation': 'This joke relies on a play on words, using cricket terminology to create a pun. \n\nIn cricket, a "wicket" refers to the set of three stumps and two bails that a batsman defends. A "sticky wicket" is a phrase used to describe a difficult or challenging situation, often originating from the idea that a wet or damp wicket can make it hard for batsmen to play their shots. \n\nThe phrase "bowled over" has a double meaning here. In cricket, a batsman can be "bowled" when the ball hits the wicket, resulting in their dismissal. However, the phrase "bowled over" is also an idiomatic expression meaning to be overwhelmed, shocked, or amazed.\n\nThe joke sets up the scenario of a cricket going to the doctor, implying that the cricket is facing some kind of problem or illness. The punchline "it had a sticky wicket situation an

In [14]:
workflow.get_state(config1) #final state values in db

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a "sticky" wicket situation and was feeling a little "bowled" over.', 'explanation': 'This joke relies on a play on words, using cricket terminology to create a pun. \n\nIn cricket, a "wicket" refers to the set of three stumps and two bails that a batsman defends. A "sticky wicket" is a phrase used to describe a difficult or challenging situation, often originating from the idea that a wet or damp wicket can make it hard for batsmen to play their shots. \n\nThe phrase "bowled over" has a double meaning here. In cricket, a batsman can be "bowled" when the ball hits the wicket, resulting in their dismissal. However, the phrase "bowled over" is also an idiomatic expression meaning to be overwhelmed, shocked, or amazed.\n\nThe joke sets up the scenario of a cricket going to the doctor, implying that the cricket is facing some kind of problem or illness. The punchline "it had a sticky 

In [ ]:
list(workflow.get_state_history(config1)) #store all intermediate values as well

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a "sticky" wicket situation and was feeling a little "bowled" over.', 'explanation': 'This joke relies on a play on words, using cricket terminology to create a pun. \n\nIn cricket, a "wicket" refers to the set of three stumps and two bails that a batsman defends. A "sticky wicket" is a phrase used to describe a difficult or challenging situation, often originating from the idea that a wet or damp wicket can make it hard for batsmen to play their shots. \n\nThe phrase "bowled over" has a double meaning here. In cricket, a batsman can be "bowled" when the ball hits the wicket, resulting in their dismissal. However, the phrase "bowled over" is also an idiomatic expression meaning to be overwhelmed, shocked, or amazed.\n\nThe joke sets up the scenario of a cricket going to the doctor, implying that the cricket is facing some kind of problem or illness. The punchline "it had a sticky

### Time Travel

In [17]:
workflow.get_state({'configurable':{'thread_id':'1','checkpoint_id':'1f12ba2d-cf21-6f5e-bfff-3e395654e5b5'}})

StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f12ba2d-cf21-6f5e-bfff-3e395654e5b5'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-03-29T19:09:51.674467+00:00', parent_config=None, tasks=(PregelTask(id='c952aa08-c98b-1ba6-e108-784337699a85', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'topic': 'pizza'}),), interrupts=())

In [18]:
workflow.invoke(None,{'configurable':{'thread_id':'1','checkpoint_id':'1f12ba2d-cf21-6f5e-bfff-3e395654e5b5'}})

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.',
 'explanation': 'The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty" is a play on words. \n\nThe phrase "feeling a little crusty" has a double meaning here. In one sense, the crust is a part of a pizza, referring to the outer layer of the bread. However, the word "crusty" also has an idiomatic meaning, which is to describe someone who is being irritable, grumpy, or having a bad temper.\n\nSo, in this joke, the punchline "it was feeling a little crusty" is a pun, using the word "crusty" to make a connection between the pizza\'s crust (the bread part) and the emotional state of being grumpy or irritable. The humor comes from the unexpected twist on the word\'s meaning, creating a clever and amusing connection between the setup and the punchline.'}

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty" is a play on words. \n\nThe phrase "feeling a little crusty" has a double meaning here. In one sense, the crust is a part of a pizza, referring to the outer layer of the bread. However, the word "crusty" also has an idiomatic meaning, which is to describe someone who is being irritable, grumpy, or having a bad temper.\n\nSo, in this joke, the punchline "it was feeling a little crusty" is a pun, using the word "crusty" to make a connection between the pizza\'s crust (the bread part) and the emotional state of being grumpy or irritable. The humor comes from the unexpected twist on the word\'s meaning, creating a clever and amusing connection between the setup and the punchline.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint

Updating State

In [23]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f12ba99-19aa-6f44-8000-b2f4cb40d004'}}

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12ba8b-622d-6d71-8000-797e6bde9d89'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-03-29T19:51:43.542165+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12ba2d-cf21-6f5e-bfff-3e395654e5b5'}}, tasks=(PregelTask(id='365e4445-296c-df83-42e5-b8cf73f332e9', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke "Why was the pizza in a bad mood? Because it was feeling a little crusty" is a play on words. \n\nThe phrase "feeling a little crusty" has a double meaning here. In one sense, the crust is a part of a pizza, referring to the o

In [ ]:
#now if want to resume from samosa topic, invoke wih the new checkpoint_id that came now, with None as topic